# Zenith v28 — GPU-Accelerated Cloud Training
## Target: 500,000 Real Human Cardiac Cells (scVI Foundation Model)
**Source:** CELLxGENE Open Data (Harmonized Sikkema et al. & PERIHEART references)

### Why run this on Google Colab?
1. **GPU Acceleration**: Training on 500k cells takes **8-12 hours** on a CPU, but completes in **20-25 minutes** on Colab's NVIDIA GPUs (like T4, L4, or A100).
2. **Zero Local Overhead**: Keeps your local PC completely cool and quiet.

### Steps to Run:
1. Create a folder in your Google Drive named `Zenith_Data`.
2. Upload your preprocessed local file `data/foundation/cardiac_preprocessed.h5ad` ($650\text{ MB}$) into that Google Drive folder.
3. Set your Colab runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`).
4. Run the code block below.

In [ ]:
# ============================================================
# STEP 1: Install dependencies
# ============================================================
!pip install scanpy scvi-tools anndata -q

# ============================================================
# STEP 2: Imports
# ============================================================
from google.colab import drive
import scanpy as sc
import scvi
import numpy as np
import torch
import os

print("GPU available:", torch.cuda.is_available())
print("scvi version:", scvi.__version__)
print("scanpy version:", sc.__version__)

# ============================================================
# STEP 3: Mount Google Drive
# ============================================================
drive.mount('/content/drive')

# ============================================================
# STEP 4: Set paths (Edit if your folder structure differs)
# ============================================================
DRIVE_PATH = "/content/drive/MyDrive/Zenith_Data/"
DATA_PATH  = DRIVE_PATH + "cardiac_preprocessed.h5ad"
MODEL_SAVE = DRIVE_PATH + "zenith_foundation_v1"

# Auto-create directory in Drive if not there
os.makedirs(DRIVE_PATH, exist_ok=True)
print("Looking for preprocessed data at:", DATA_PATH)
print("Data file exists:", os.path.exists(DATA_PATH))

# ============================================================
# STEP 5: Load 500k preprocessed cells into RAM
# ============================================================
if os.path.exists(DATA_PATH):
    print("Loading 500,000 preprocessed cells into RAM...")
    adata = sc.read_h5ad(DATA_PATH)
    print(adata)
    
    # Cast covariates to category to ensure scVI compiles correctly
    adata.obs['dataset_id'] = adata.obs['dataset_id'].astype('category')
    adata.obs['suspension_type'] = adata.obs['suspension_type'].astype('category')

    # ============================================================
    # STEP 6: Setup scVI VAE model
    # ============================================================
    print("\nConfiguring scVI model...")
    scvi.model.SCVI.setup_anndata(
        adata,
        layer="counts",
        batch_key="dataset_id",
        categorical_covariate_keys=["suspension_type"]
    )
    
    model = scvi.model.SCVI(
        adata,
        n_hidden=256,
        n_latent=30,
        n_layers=2,
        gene_likelihood="nb",
        use_layer_norm="both"
    )
    
    print("\nModel ready. Starting GPU-accelerated training on", adata.n_obs, "cells...")
    
    # ============================================================
    # STEP 7: Train on Cloud GPU (~20-25 minutes on T4 GPU)
    # ============================================================
    use_gpu = torch.cuda.is_available()
    model.train(
        max_epochs=400,
        early_stopping=True,
        early_stopping_patience=30,
        early_stopping_monitor="elbo_validation",
        check_val_every_n_epoch=5,
        train_size=0.90,
        batch_size=512,  # Optimized batch size for GPU parallelization
        accelerator='gpu' if use_gpu else 'cpu'
    )
    
    # ============================================================
    # STEP 8: Save model and var schema to Google Drive
    # ============================================================
    print("\nSaving trained model to Google Drive...")
    model.save(MODEL_SAVE, overwrite=True)
    
    # Save the var schema (required for future patient biopsy alignments)
    ref_adata = sc.AnnData(
        X   = np.zeros((1, adata.n_vars), dtype=np.float32),
        var = adata.var.copy(),
        dtype=np.float32
    )
    ref_adata.write_h5ad(os.path.join(MODEL_SAVE, "var_schema.h5ad"))
    
    print("\n✅ SUCCESS! Zenith Foundation Model v28 trained and saved.")
    print("Download the 'zenith_foundation_v1' folder from Google Drive and replace your local 'models/zenith_foundation_v1/' folder.")
else:
    print("❌ ERROR: Please upload 'cardiac_preprocessed.h5ad' (located in your local 'data/foundation/' directory) to your Google Drive folder 'Zenith_Data/' first!")